In [12]:
pip install langchain langchain-community langchain-core langchain-huggingface faiss-cpu sentence-transformers fastapi uvicorn ollama

Note: you may need to restart the kernel to use updated packages.


In [4]:
!pip install transformers torch sentencepiece faiss-cpu PyPDF2 tqdm numpy

  Using cached pypdf2-3.0.1-py3-none-any.whl.metadata (6.8 kB)
Using cached pypdf2-3.0.1-py3-none-any.whl (232 kB)


In [3]:
import ollama
Prompt = input("Enter the query :")
response = ollama.chat(
    model="llama3:8b",
    messages=[
        {"role": "user", "content":Prompt }
    ]
)

print(response["message"]["content"])

Enter the query : what is ai and diff between ai and ml 


**What is AI?**

Artificial Intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as:

1. Learning: AI systems can learn from data and improve their performance over time.
2. Reasoning: AI systems can draw conclusions based on available information.
3. Problem-solving: AI systems can find solutions to complex problems.

AI involves a range of techniques, including machine learning (ML), natural language processing (NLP), computer vision, robotics, and expert systems.

**What is ML?**

Machine Learning (ML) is a subset of AI that enables computers to learn from data without being explicitly programmed. In other words, ML algorithms analyze patterns in data and make predictions or decisions based on that analysis.

ML involves training models using large datasets and adjusting the model's parameters to improve its performance over time. There are many types of ML algorithms, including:

1. Supervised learnin

In [54]:
import os
import re
import pickle
import numpy as np
import faiss
import PyPDF2

from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

DATA_FOLDER = r"D:\data"
TXT_FOLDER = os.path.join(DATA_FOLDER, "txt")
PDF_FOLDER = os.path.join(DATA_FOLDER, "pdf")

OUTPUT_FOLDER = r"D:\custom_embeddings"

INDEX_FILE = os.path.join(OUTPUT_FOLDER, "index.faiss")
META_FILE = os.path.join(OUTPUT_FOLDER, "metadata.pkl")

os.makedirs(OUTPUT_FOLDER, exist_ok=True)


class EnterpriseSemanticEmbeddingPipeline:

    def __init__(self):

        print("Loading embedding model...")

        self.model = SentenceTransformer(
            "sentence-transformers/all-MiniLM-L6-v2"
        )

        self.dim = self.model.get_sentence_embedding_dimension()

        print("Embedding dimension:", self.dim)

        self.index, self.metadata = self.load_index()

        self.abbreviations = {

            "llm": "large language model",
            "ml": "machine learning",
            "ai": "artificial intelligence",
            "nlp": "natural language processing",
            "cnn": "convolutional neural network",
            "rnn": "recurrent neural network",
            "bert": "bidirectional encoder representations from transformers"

        }

    def load_index(self):

        if os.path.exists(INDEX_FILE):

            print("Loading existing FAISS index...")

            index = faiss.read_index(INDEX_FILE)

            with open(META_FILE, "rb") as f:
                metadata = pickle.load(f)

        else:

            print("Creating new FAISS index...")

            index = faiss.IndexFlatIP(self.dim)

            metadata = []

        return index, metadata


    def clean_text(self, text):

        text = re.sub(r"http\S+", "", text)
        text = re.sub(r"\[\d+\]", "", text)
        text = re.sub(r"\s+", " ", text)

        return text.strip()

    def split_sentences(self, text):

        sentences = re.split(r'(?<=[.!?])\s+', text)

        clean = []

        for s in sentences:

            s = s.strip()

            if len(s) < 30:
                continue

            clean.append(s)

        return clean
    def semantic_chunk(self, sentences,
                       chunk_size=1000,
                       overlap=600):

        chunks = []

        current_chunk = []
        current_length = 0

        for sentence in sentences:

            words = sentence.split()
            length = len(words)

            if current_length + length > chunk_size:

                chunks.append(" ".join(current_chunk))

                # overlap preservation
                overlap_words = " ".join(current_chunk).split()[-overlap:]

                current_chunk = [" ".join(overlap_words)]

                current_length = len(overlap_words)

            current_chunk.append(sentence)
            current_length += length

        if current_chunk:
            chunks.append(" ".join(current_chunk))

        return chunks

    def process_txt(self):

        data = []

        print("Processing TXT files...")

        for root, _, files in os.walk(TXT_FOLDER):

            for file in files:

                if not file.endswith(".txt"):
                    continue

                path = os.path.join(root, file)

                with open(path, encoding="utf-8") as f:
                    text = f.read()

                text = self.clean_text(text)

                sentences = self.split_sentences(text)

                chunks = self.semantic_chunk(sentences)

                for chunk in chunks:

                    data.append({
                        "text": chunk,
                        "source": path,
                        "type": "txt"
                    })

                    # abbreviation expansion
                    for abbr, full in self.abbreviations.items():

                        if abbr in chunk.lower():

                            expanded = chunk.replace(abbr, full)

                            data.append({
                                "text": expanded,
                                "source": path,
                                "type": "txt"
                            })

        return data

    def process_pdf(self):

        data = []

        print("Processing PDF files...")

        for root, _, files in os.walk(PDF_FOLDER):

            for file in files:

                if not file.endswith(".pdf"):
                    continue

                path = os.path.join(root, file)

                text = ""

                with open(path, "rb") as f:

                    reader = PyPDF2.PdfReader(f)

                    for page in reader.pages:

                        page_text = page.extract_text()

                        if page_text:
                            text += page_text + " "

                text = self.clean_text(text)

                sentences = self.split_sentences(text)

                chunks = self.semantic_chunk(sentences)

                for chunk in chunks:

                    data.append({
                        "text": chunk,
                        "source": path,
                        "type": "pdf"
                    })

                    for abbr, full in self.abbreviations.items():

                        if abbr in chunk.lower():

                            expanded = chunk.replace(abbr, full)

                            data.append({
                                "text": expanded,
                                "source": path,
                                "type": "pdf"
                            })

        return data

    def create_embeddings(self, data):

        print("Creating embeddings...")

        texts = [x["text"] for x in data]

        embeddings = self.model.encode(
            texts,
            batch_size=64,
            normalize_embeddings=True,
            show_progress_bar=True
        )

        return np.array(embeddings).astype("float32")

    def save(self):

        faiss.write_index(self.index, INDEX_FILE)

        with open(META_FILE, "wb") as f:
            pickle.dump(self.metadata, f)


    def run(self):

        txt_data = self.process_txt()

        pdf_data = self.process_pdf()

        all_data = txt_data + pdf_data

        print("Total chunks:", len(all_data))

        embeddings = self.create_embeddings(all_data)

        self.index.add(embeddings)

        self.metadata.extend(all_data)

        self.save()

        print("Embedding complete.")


if __name__ == "__main__":

    pipeline = EnterpriseSemanticEmbeddingPipeline()

    pipeline.run()

Loading embedding model...
Embedding dimension: 384
Creating new FAISS index...
Processing TXT files...
Processing PDF files...
Total chunks: 845
Creating embeddings...


Batches:   0%|          | 0/14 [00:00<?, ?it/s]

Embedding complete.


In [55]:

import os
import re
import pickle
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer, CrossEncoder
from difflib import get_close_matches


EMBEDDING_PATH = r"D:\custom_embeddings"

INDEX_FILE = os.path.join(EMBEDDING_PATH, "index.faiss")
METADATA_FILE = os.path.join(EMBEDDING_PATH, "metadata.pkl")

TOP_K = 50
FINAL_TOP_K = 8
SENTENCE_TOP_K = 8

VECTOR_WEIGHT = 0.6
RERANK_WEIGHT = 0.4




print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cpu"
)

DIM = embedding_model.get_sentence_embedding_dimension()

print("Embedding dimension:", DIM)


print("Loading reranker...")

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    device="cpu"
)


print("Loading FAISS index...")

index = faiss.read_index(INDEX_FILE)


print("Loading metadata...")

with open(METADATA_FILE, "rb") as f:
    metadata = pickle.load(f)

print("System ready.\n")


VOCAB = set()

for item in metadata:

    words = re.findall(r"[a-zA-Z]{3,}", item["text"].lower())

    VOCAB.update(words)


def correct_spelling(query):

    corrected = []

    for word in query.split():

        match = get_close_matches(word, VOCAB, n=1, cutoff=0.80)

        if match:
            corrected.append(match[0])
        else:
            corrected.append(word)

    return " ".join(corrected)


ABBREVIATIONS = {

    "llm": "large language model",
    "ml": "machine learning",
    "ai": "artificial intelligence",
    "nlp": "natural language processing",
    "cnn": "convolutional neural network",
    "rnn": "recurrent neural network",
    "bert": "bidirectional encoder representations from transformers"
}


def normalize_query(query):

    query = query.lower()

    query = correct_spelling(query)

    words = query.split()

    expanded_words = []

    for word in words:

        if word in ABBREVIATIONS:
            expanded_words.append(ABBREVIATIONS[word])
        else:
            expanded_words.append(word)

    query = " ".join(expanded_words)

    query = re.sub(r"[^a-z0-9\s]", " ", query)

    query = re.sub(r"\s+", " ", query)

    return query.strip()


def expand_query(query):

    expanded = set()

    expanded.add(query)

    expanded.add(correct_spelling(query))

    expanded.add("what is " + query)

    expanded.add(query + " definition")

    expanded.add(query + " explanation")

    expanded.add(query + " meaning")

    words = query.split()

    for word in words:

        if word in ABBREVIATIONS:

            expanded.add(ABBREVIATIONS[word])

    return list(expanded)



def vector_search(query):

    expanded_queries = expand_query(query)

    vectors = embedding_model.encode(
        expanded_queries,
        normalize_embeddings=True
    ).astype("float32")

    all_results = {}

    for vec in vectors:

        scores, indices = index.search(vec.reshape(1, -1), TOP_K)

        for idx, score in zip(indices[0], scores[0]):

            if idx == -1:
                continue

            if idx not in all_results or score > all_results[idx]["vector_score"]:

                all_results[idx] = {

                    "text": metadata[idx]["text"],
                    "vector_score": float(score)
                }

    return list(all_results.values())


def rerank(query, results):

    if not results:
        return []

    pairs = [(query, r["text"]) for r in results]

    scores = reranker.predict(pairs)

    for r, score in zip(results, scores):

        r["rerank_score"] = float(score)

        r["final_score"] = (

            VECTOR_WEIGHT * r["vector_score"]
            +
            RERANK_WEIGHT * r["rerank_score"]

        )

    results.sort(

        key=lambda x: x["final_score"],
        reverse=True

    )

    return results[:FINAL_TOP_K]


DEFINITION_PATTERNS = [

    " is ",
    " refers to ",
    " defined as ",
    " stands for "

]


def extract_best_sentences(query, contexts):

    query_emb = embedding_model.encode(
        query,
        normalize_embeddings=True
    )

    candidates = []

    seen = set()

    for ctx in contexts:

        sentences = re.split(r"[.?!]\s+", ctx["text"])

        for s in sentences:

            s = s.strip()

            if len(s) < 30:
                continue

            if s in seen:
                continue

            seen.add(s)

            if any(x in s.lower() for x in [
                "chapter",
                "figure",
                "table",
                "page",
                "section"
            ]):
                continue

            s_emb = embedding_model.encode(
                s,
                normalize_embeddings=True
            )

            sim = float(np.dot(query_emb, s_emb))

            definition_bonus = 0.20 if any(
                p in s.lower()
                for p in DEFINITION_PATTERNS
            ) else 0

            final_score = sim + definition_bonus

            candidates.append(
                (s, final_score)
            )

    candidates.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return candidates[:SENTENCE_TOP_K]


def build_answer(query, contexts):

    if not contexts:
        return "No reliable answer found in knowledge base."

    best = extract_best_sentences(query, contexts)

    if not best:
        return contexts[0]["text"]

    definition = best[0][0]

    explanation = [s for s, score in best[1:]]

    answer = definition + ".\n\n"

    if explanation:

        answer += "Explanation:\n\n"

        for s in explanation[:5]:

            answer += "• " + s + ".\n"

    return answer


def answer_query(query):

    normalized_query = normalize_query(query)

    retrieved = vector_search(normalized_query)

    reranked = rerank(normalized_query, retrieved)

    answer = build_answer(normalized_query, reranked)

    return answer


def ask():

    while True:

        query = input("\nEnter query (or exit): ")

        if query.lower() == "exit":
            break

        answer = answer_query(query)

        print("\nAnswer:\n")
        print(answer)


if __name__ == "__main__":

    ask()

Loading embedding model...
Embedding dimension: 384
Loading reranker...
Loading FAISS index...
Loading metadata...
System ready.




Enter query (or exit):  llm



Answer:

The “large” in “large language model” refers to both the model’s size in terms of parameters and the immense dataset on which it’s trained.

Explanation:

• Performance Metrics of Large Language Models: The performance of LLMs is evaluated using several metrics.
• A Large Language Model (LLM) is a type of artificial intelligence system designed to process, understand, and generate human-like language.
• Perplexity is the most important metric for language models.
• Conclusion : Large Language Models represent one of the most significant advancements in artificial intelligence.
• Working of Large Language Models: Large Language Models are primarily based on the transformer architecture, which was introduced in 2017.




Enter query (or exit):  Artificial intelligence 



Answer:

Artificial Intelligence (AI) is a branch of computer science that focuses on designing and developing systems capable of performing tasks that typically require human intelligence.

Explanation:

• Conclusion Artificial Intelligence is a multidisciplinary field that enables machines to simulate human intelligence using algorithms, data, and computational models.
• Machine Learning (ML) is a subset of AI that enables systems to learn from data automatically.
• The fundamental objective of AI is to create systems that can function autonomously, adapt to new situations, and improve their performance over time through experience.
• Trartificial intelligencening is the process of teaching the AI model using data.
• Learning is one of the most important components of AI.




Enter query (or exit):  Machine Learning



Answer:

Machine Learning (ML) is a subset of AI that enables systems to learn from data automatically.

Explanation:

• Machine learning is a subcategory of AI that is concerned with algorithms that learn from data.
• In supervised learning, models are trartificial intelligencened using labeled data, where each input is associated with a known output.
• Using a learning algorithm, a model is trartificial intelligencened on a trartificial intelligencening dataset consisting of examples and corresponding labels.
• Using a learning algorithm, a model is trained on a training dataset consisting of examples and corresponding labels.
• In supervised learning, models are trained using labeled data, where each input is associated with a known output.



KeyboardInterrupt: Interrupted by user